# 02 -- Prepare Confounder Dataset

**Purpose:** Download images for all four hard-negative confounder categories and
re-run the stratified train/val/test split to incorporate them into
`processed_data/binary/`.

**Target:** 400 images per category (gives ~60 test images after 70/15/15 split,
yielding a Clopper-Pearson CI upper bound of ~5.9 % at 0 % FP rate — statistically
defensible for *Computers & Geosciences*).

**Categories:**
| Category | Source(s) | Status |
|---|---|---|
| Swimmingpool | Places365 + Open Images v7 | supplement to 400 |
| River | ATLANTIS + RIWA + WaterNet + LuFI-RiverSnap | new |
| Lake | ATLANTIS + WaterNet | new |
| Fountain | Open Images v7 + ADE20K | new |

**Compute:** CPU only — no GPU required for downloading or splitting.

**Estimated runtime:** 10–60 minutes (dominated by download speed).

**Prerequisites:**
- `pip install requests Pillow tqdm` (already in project environment)
- Kaggle sources (RIWA, WaterNet, LuFI-RiverSnap): place `~/.kaggle/kaggle.json`
- Open Images fiftyone path (optional): `pip install fiftyone` — falls back to CSV if absent

**Outputs:**
```
data/FloodingDataset2/junk/{Category}/{Category}_NNNN.jpg   ← raw downloads
data/FloodingDataset2/train/non_flood/                      ← updated split
data/FloodingDataset2/val/non_flood/
data/FloodingDataset2/test/non_flood/
data/FloodingDataset2/split_manifest.csv                    ← audit trail
```

In [ ]:
# Mount Google Drive and set working directory
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/imagevalidation2

In [ ]:
# Verify required packages are available
import importlib, sys

REQUIRED = ['requests', 'PIL', 'tqdm']
OPTIONAL = ['kaggle', 'fiftyone']

for pkg in REQUIRED:
    if importlib.util.find_spec(pkg) is None:
        raise ImportError(f'Missing required package: {pkg}  →  pip install {pkg}')
    print(f'  [OK] {pkg}')

for pkg in OPTIONAL:
    found = importlib.util.find_spec(pkg) is not None
    label = 'OK' if found else 'MISSING (optional — relevant sources will be skipped)'
    print(f'  [{label}] {pkg}')

## Step 1 — Download confounder images

Each script is idempotent: it counts existing `{Category}_*.jpg` files and only
downloads the difference to reach `--max`.  Re-running after a partial failure is safe.

In [ ]:
# 1a: Supplement Swimmingpool to 400 images
!python scripts/download_swimmingpool.py \
    --output_dir ./data/FloodingDataset2/junk \
    --max 400

In [ ]:
# 1b: Download River images (new category)
!python scripts/download_river.py \
    --output_dir ./data/FloodingDataset2/junk \
    --max 400

In [ ]:
# 1c: Download Lake images (new category)
!python scripts/download_lake.py \
    --output_dir ./data/FloodingDataset2/junk \
    --max 400

In [ ]:
# 1d: Download Fountain images (new category)
!python scripts/download_fountain.py \
    --output_dir ./data/FloodingDataset2/junk \
    --max 400

In [ ]:
# Verify download counts per category
from pathlib import Path

junk_dir = Path('./data/FloodingDataset2/junk')
NEW_CATS = ['Swimmingpool', 'River', 'Lake', 'Fountain']

print(f"{'Category':<20}  {'Count':>6}  {'Status'}")
print('-' * 45)
for cat in sorted(junk_dir.iterdir()):
    if not cat.is_dir():
        continue
    n = len(list(cat.glob('*.jpg')))
    flag = ' ← NEW' if cat.name in NEW_CATS else ''
    status = '✓ on target' if n >= 400 else f'⚠ only {n}'
    if cat.name in NEW_CATS:
        status = ('✓ on target' if n >= 400 else f'⚠ only {n}')
    print(f'{cat.name:<20}  {n:>6}  {status}{flag}')

## Step 2 — Stratified 70 / 15 / 15 split

Scans all `junk/{Category}/` directories, applies a seeded 70/15/15 split **per category**
(stratified so every category is represented in all three partitions), and writes symlinks
into the binary dataset structure.

**This replaces the existing `train/non_flood/`, `val/non_flood/`, `test/non_flood/`
contents** — run it once after all download scripts have completed.

> *The split is deterministic: seeded with `SPLIT_SEED = 42`. Changing the seed
> would change all assignments, invalidating previously trained checkpoints.*

In [ ]:
import shutil
import numpy as np
import pandas as pd
from pathlib import Path

# ── Configuration ──────────────────────────────────────────────────────────
DATA_DIR    = Path('./data/FloodingDataset2')
JUNK_DIR    = DATA_DIR / 'junk'
SPLIT_SEED  = 42
VAL_FRAC    = 0.15
TEST_FRAC   = 0.15
# ───────────────────────────────────────────────────────────────────────────

rng = np.random.default_rng(seed=SPLIT_SEED)

split_dirs = {
    'train': DATA_DIR / 'train' / 'non_flood',
    'val':   DATA_DIR / 'val'   / 'non_flood',
    'test':  DATA_DIR / 'test'  / 'non_flood',
}

# Clear existing non_flood splits and rebuild from junk
for split_dir in split_dirs.values():
    if split_dir.exists():
        shutil.rmtree(split_dir)
    split_dir.mkdir(parents=True, exist_ok=True)

records = []

for cat_dir in sorted(JUNK_DIR.iterdir()):
    if not cat_dir.is_dir():
        continue
    images = sorted(cat_dir.glob('*.jpg'))  # sorted → deterministic
    if not images:
        print(f'  [SKIP] {cat_dir.name}: no .jpg files')
        continue

    n = len(images)
    indices = rng.permutation(n)  # seeded shuffle

    n_val  = max(1, round(n * VAL_FRAC))
    n_test = max(1, round(n * TEST_FRAC))

    val_idx   = indices[:n_val]
    test_idx  = indices[n_val:n_val + n_test]
    train_idx = indices[n_val + n_test:]

    for split_name, idx_arr in [('train', train_idx), ('val', val_idx), ('test', test_idx)]:
        dest_dir = split_dirs[split_name]
        for i in idx_arr:
            src = images[i]
            dst = dest_dir / src.name
            shutil.copy2(src, dst)
            records.append({'category': cat_dir.name, 'split': split_name, 'filename': src.name})

    print(f'  {cat_dir.name:<20}  n={n:>4}  '
          f'train={len(train_idx):>3}  val={len(val_idx):>3}  test={len(test_idx):>3}')

# Save split manifest for reproducibility audit
manifest = pd.DataFrame(records)
manifest.to_csv(DATA_DIR / 'split_manifest.csv', index=False)
print(f'\nManifest saved → {DATA_DIR / "split_manifest.csv"}  ({len(manifest)} rows)')

In [ ]:
# Verify final split counts and display summary table
import pandas as pd
from pathlib import Path

manifest = pd.read_csv('./data/FloodingDataset2/split_manifest.csv')

summary = (
    manifest
    .groupby(['category', 'split'])
    .size()
    .unstack(fill_value=0)
    [['train', 'val', 'test']]
)
summary['total'] = summary.sum(axis=1)
summary['test%'] = (summary['test'] / summary['total'] * 100).round(1)

print('Non-flood split summary (all categories):')
print(summary.to_string())
print(f"\nTotal non-flood images: {summary['total'].sum()}")
print(f"  train: {summary['train'].sum()}  val: {summary['val'].sum()}  test: {summary['test'].sum()}")

## Next Steps

With the split in place, re-run the full pipeline:

1. **Notebooks 03 / 04** — Re-run baseline training if you want the model to see the new
   categories during training (recommended for final paper results).
2. **Notebook 05a** — Re-run confounder analysis to see FP rates for River, Lake, Fountain.
3. **Notebooks 05b / 05c** — Re-run HNM with the updated confounder candidate list.
4. **Notebook 06** — Re-run evaluation; the per-category FP table now includes all four
   visual confounder categories with Clopper-Pearson CIs.